# Notebook 2 — Data Cleaning: BudgetWise Synthetic Dirty Dataset
**File:** `budgetwise_synthetic_dirty.csv`  
**Goal:** Clean the raw dataset and export a clean CSV ready for combining in Notebook 3  
**Tool:** Python (Pandas)

---
### Data Quality Issues Found
- Mixed date formats including full month names e.g. `December 22 2021`
- Category column has 200+ variations of just 12 valid categories
- Payment mode has 60+ variations of just 4 valid values
- Amount stored as string with `$` signs, commas and sentinel values like `999,999`
- Location has abbreviations, mixed casing and inconsistent naming
- Missing values across date, category, amount, payment_mode and location
- Duplicate rows and duplicate transaction IDs
- Junk entries in notes column

## Step 1: Import Libraries

In [54]:
import pandas as pd
import numpy as np
import re
import sys



## Step 2: Load the Raw Dataset

In [55]:
df = pd.read_csv('C:/Users/Chibueze/Valentine-Python-Project/PythonProject/Practical/BudgetWise Personal Finance Dataset/data/raw/budgetwise_synthetic_dirty.csv')

df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T03512,U039,December 22 2021,Expense,Rent,998,Cash,Pune,Paid electricity bill
1,T03261,U179,03/24/2022,Expense,Food,$143,Card,Delhi,Grocery shopping
2,T04316,U143,October 18 2022,Expense,Rent,149,Cash,Bengaluru,NaN
3,T05649,U079,12/12/2021,Expense,Rent,49,UPI,NaN,Paid electricity bill
4,T14750,U020,NaN,Income,Other Income,"83,802",Bank Transfer,Chennai,Gift via app
...,...,...,...,...,...,...,...,...,...
15831,T05700,U101,June 15 2019,Expense,Food,470,UPI,Pune,Uber to office
15832,T10743,U001,01/11/2022,Expense,Food,₹432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit
15834,T09413,U005,07/10/2021,Expense,Rent,₹892,UPI,Hyderabad,Dinner at resto


In [56]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15836 entries, 0 to 15835
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    15836 non-null  object
 1   user_id           15836 non-null  object
 2   date              15492 non-null  object
 3   transaction_type  15836 non-null  object
 4   category          15678 non-null  object
 5   amount            15658 non-null  object
 6   payment_mode      15333 non-null  object
 7   location          15114 non-null  object
 8   notes             14302 non-null  object
dtypes: object(9)
memory usage: 1.1+ MB


,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
count,15836,15836,15492,15836,15678,15658,15333,15114,14302
unique,14858,192,5319,2,212,6279,62,30,988
top,T14837,U020,2020-08-29,Expense,Food,"999,999",Bank Transfer,Pune,Gift
freq,3,105,12,13442,3764,38,3838,1416,980


## Step 3: Initial Data Overview

In [57]:
# Check data types

print(' Data Types ')
print(df.dtypes)

print('\n Missing Values ')
print(df.isnull().sum())

print('\n Missing Values % ')
print((df.isnull().sum() / len(df) * 100).round(2))

 Data Types 
transaction_id      object
user_id             object
date                object
transaction_type    object
category            object
amount              object
payment_mode        object
location            object
notes               object
dtype: object

 Missing Values 
transaction_id         0
user_id                0
date                 344
transaction_type       0
category             158
amount               178
payment_mode         503
location             722
notes               1534
dtype: int64

 Missing Values % 
transaction_id      0.00
user_id             0.00
date                2.17
transaction_type    0.00
category            1.00
amount              1.12
payment_mode        3.18
location            4.56
notes               9.69
dtype: float64


## Step 4: Replace Fake Null Strings
**Issue:** Some missing values are stored as text strings like `'Nan'`, `'N/A'`, `'NULL'` instead of real NaN

In [58]:
# Replace all common fake null strings across entire dataframe at once
df = df.replace(['Nan', 'NaN', 'nan', 'NULL', 'null', 'N/A', 'na', 'none', 'None', '', ' '], np.nan)

print('Missing values after replacing fake nulls:')
print(df.isnull().sum())

Missing values after replacing fake nulls:
transaction_id         0
user_id                0
date                 344
transaction_type       0
category             158
amount               178
payment_mode         503
location             722
notes               1534
dtype: int64


## Step 5: Remove Duplicate Rows

In [59]:
# Remove fully duplicate rows first

df = df.drop_duplicates()

In [60]:
# Remove duplicate transaction IDs and keep first occurrence

df = df.drop_duplicates(subset='transaction_id', keep='first' )

In [61]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T03512,U039,December 22 2021,Expense,Rent,998,Cash,Pune,Paid electricity bill
1,T03261,U179,03/24/2022,Expense,Food,$143,Card,Delhi,Grocery shopping
2,T04316,U143,October 18 2022,Expense,Rent,149,Cash,Bengaluru,NaN
3,T05649,U079,12/12/2021,Expense,Rent,49,UPI,NaN,Paid electricity bill
4,T14750,U020,NaN,Income,Other Income,"83,802",Bank Transfer,Chennai,Gift via app
...,...,...,...,...,...,...,...,...,...
15831,T05700,U101,June 15 2019,Expense,Food,470,UPI,Pune,Uber to office
15832,T10743,U001,01/11/2022,Expense,Food,₹432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit
15834,T09413,U005,07/10/2021,Expense,Rent,₹892,UPI,Hyderabad,Dinner at resto


## Step 6: Clean the Date Column
**Issues:** Multiple mixed formats including full month names:- `December 22 2021`, `03/24/2022`, `04-11-22`, `2022-01-06`

In [62]:
df['date'].unique()

array(['December 22 2021', '03/24/2022', 'October 18 2022', ...,
       'June 16 2021', 'August 09 2019', '07/10/2021'], dtype=object)

In [63]:
print('Sample date values before cleaning:')
print(df['date'].dropna().unique()[:15])

Sample date values before cleaning:
['December 22 2021' '03/24/2022' 'October 18 2022' '12/12/2021' '12-07-22'
 'September 12 2019' '2022-01-06' '04-11-22' 'February 03 2021' '02-11-22'
 '11/21/2022' '12/29/2022' '2020-10-26' '2021-05-25' 'March 03 2021']


In [64]:
# Custom parser to handle all date formats including full month names
def parse_mixed_dates(date_str):
    if pd.isnull(date_str):
        return pd.NaT

    formats = [
        '%Y-%m-%d',        # 2022-01-06
        '%m/%d/%Y',        # 03/24/2022
        '%d/%m/%Y',        # 24/03/2022
        '%d-%m-%y',        # 04-11-22
        '%d-%m-%Y',        # 04-11-2022
        '%m-%d-%Y',        # 03-24-2022
        '%B %d %Y',        # December 22 2021
        '%b %d %Y',        # Dec 22 2021
        '%d %B %Y',        # 22 December 2021
        '%Y/%m/%d',        # 2022/01/06
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    return pd.NaT

df['date'] = df['date'].apply(parse_mixed_dates)

print('\nUnparseable dates (NaT):', df['date'].isna().sum())


Unparseable dates (NaT): 339


In [65]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill
1,T03261,U179,2022-03-24,Expense,Food,$143,Card,Delhi,Grocery shopping
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bengaluru,NaN
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,NaN,Paid electricity bill
4,T14750,U020,NaT,Income,Other Income,"83,802",Bank Transfer,Chennai,Gift via app
...,...,...,...,...,...,...,...,...,...
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office
15832,T10743,U001,2022-01-11,Expense,Food,₹432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit
15834,T09413,U005,2021-07-10,Expense,Rent,₹892,UPI,Hyderabad,Dinner at resto


In [66]:
# Drop rows where date could not be parsed cause date is critical for trend analysis

df = df.dropna(subset=['date']).copy()

print('Rows after dropping missing dates:', len(df))

Rows after dropping missing dates: 14519


In [67]:
# Standardise to YYYY-MM-DD string format

df['date'] = df['date'].dt.strftime('%Y-%m-%d')

In [68]:
# Add helper columns for SQL and Power BI

df['year'] = pd.to_datetime(df['date']).dt.year
df['month'] = pd.to_datetime(df['date']).dt.month
df['month_name'] = pd.to_datetime(df['date']).dt.strftime('%B')

print('\nSample dates after cleaning:')
print(df['date'].unique()[:10])



Sample dates after cleaning:
['2021-12-22' '2022-03-24' '2022-10-18' '2021-12-12' '2022-07-12'
 '2019-09-12' '2022-01-06' '2022-11-04' '2021-02-03' '2022-11-02']


In [69]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,$143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bengaluru,NaN,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,NaN,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
...,...,...,...,...,...,...,...,...,...,...,...,...
15830,T04308,U052,2020-10-15,Expense,Food,140,Cash,Bengaluru,NaN,2020,10,October
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office,2019,6,June
15832,T10743,U001,2022-01-11,Expense,Food,₹432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta,2022,1,January
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit,2022,12,December


## Step 7: Clean the Amount Column
**Issues:** Stored as string, `$` signs, commas, sentinel value `999,999`, missing values

In [70]:
df['amount'].unique()

array(['998', '$143', '149', ..., '₹1,324', '52,674', '₹432'],
      dtype=object)

In [71]:
print('Sample Amount column before cleaning:')
print(df['amount'].value_counts().head(15))

Sample Amount column before cleaning:
amount
999,999    38
103        30
106        27
133        26
76         26
116        25
184        24
195        24
152        24
113        23
50         23
164        23
89         23
144        23
78         22
Name: count, dtype: int64


In [72]:
# Remove currency symbols and commas

df['amount'] = df['amount'].astype(str).str.replace('$', '',regex=False)

df['amount'] = df['amount'].str.replace('₹', '',regex=False)

df['amount'] = df['amount'].str.replace(',', '',regex=False)

df['amount'] = df['amount'].str.strip()

In [73]:
# Convert to numeric and non-numeric values become NaN

df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

In [74]:
# Check impossible values
print('\nRows with negative amounts:', (df['amount'] < 0).sum())
print('Rows with zero amount:', (df['amount'] == 0).sum())
print('Rows with sentinel value 999999:', (df['amount'] == 999999).sum())


Rows with negative amounts: 59
Rows with zero amount: 0
Rows with sentinel value 999999: 43


In [75]:
# Remove impossible and sentinel values
df = df[df['amount'] > 0].copy()

df = df[df['amount'] <= 500000].copy()

In [76]:
# Drop rows where amount is still null

df = df.dropna(subset=['amount']).copy()

In [77]:
# Cast to integer

df['amount'] = df['amount'].astype(int, errors='ignore')

print('\nAmount after cleaning:')
print(df['amount'].describe())
print('\nRows remaining:', len(df))


Amount after cleaning:
count     14259.000000
mean       9789.188583
std       21938.372183
min           4.000000
25%         205.000000
50%         533.000000
75%        1725.500000
max      105562.000000
Name: amount, dtype: float64

Rows remaining: 14259


In [78]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bengaluru,NaN,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,NaN,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
...,...,...,...,...,...,...,...,...,...,...,...,...
15830,T04308,U052,2020-10-15,Expense,Food,140,Cash,Bengaluru,NaN,2020,10,October
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office,2019,6,June
15832,T10743,U001,2022-01-11,Expense,Food,432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta,2022,1,January
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit,2022,12,December


## Step 8: Clean the Category Column
**Issues:** 200+ variations of just 12 valid categories with heavy typos, double letters, scrambled letters

In [79]:
df['category'].unique()

array(['Rent', 'Food', 'Entertainment', 'Savings', 'Education', 'Others',
       'Entertainmennt', 'Salaryy', 'Other Income', 'Travel', 'Reent',
       'Bonus', 'Health', 'Salary', 'Utilities', nan, 'ood', 'Traavel',
       'Tavel', 'OOthers', 'Utilites', 'eRnt', 'Trave', 'Savingss',
       'rTavel', 'Foo', 'Otehr Income', 'Rnt', 'Otehrs', 'Retn',
       'Heealth', 'Fod', 'Foodd', 'Traveel', 'Rennt', 'Ret', 'Foood',
       'Othhers', 'aSvings', 'Utilitise', 'Travle', 'Healt', 'FFood',
       'Bonu', 'BBonus', 'Bons', 'oFod', 'Eduation', 'Entertainmentt',
       'Otherrs', 'Salry', 'Saary', 'Utilitiees', 'Entertainemnt',
       'tOhers', 'Utiilties', 'RRent', 'ravel', 'Sallary', 'Savnigs',
       'Trravel', 'Utiliies', 'Heaalth', 'Rnet', 'Saviings', 'Bonuus',
       'ent', 'Saivngs', 'Savngs', 'Entetainment', 'Svings', 'Ren',
       'Salaary', 'Entertaniment', 'Utilitis', 'Bouns', 'Utiltiies',
       'Oters', 'aSlary', 'tOher Income', 'Enntertainment', 'Fodo',
       'Edcation', 'Trael'

In [80]:
print('Unique category count before cleaning:', df['category'].nunique())
print('\nCategory values before cleaning:')
print(df['category'].value_counts().to_string())

Unique category count before cleaning: 210

Category values before cleaning:
category
Food              3368
Rent              2783
Travel            1324
Utilities         1155
Entertainment      960
Bonus              679
Salary             678
Other Income       672
Others             544
Savings            523
Education          305
Health             213
Fod                 46
Foood               41
FFood               28
eRnt                22
Fodo                22
Ret                 22
Rentt               21
Foo                 20
Reent               20
Retn                19
oFod                19
Foodd               18
Ren                 17
ood                 16
ent                 16
Rnet                15
Rennt               14
Rnt                 14
RRent               14
Tavel               12
Trave                9
Utilitiees           9
Tarvel               8
ravel                8
Travle               7
rTavel               7
Trvael               7
aSvings          

In [ ]:
# Write all unique category values to a text file
with open('category_values.txt', 'w', encoding='utf-8') as f:
    f.write(f'Total unique values: {df["category"].nunique()}\n\n')
    for val, count in df['category'].value_counts().items():
        f.write(f'{val}: {count}\n')

print('Done — open category_values.txt in your folder')

In [81]:
# Strip whitespace and standardise casing first

df['category'] = df['category'].astype(str).str.strip().str.title()

In [82]:
# Use fuzzy keyword matching to catch all variations
def standardise_category(val):
    if pd.isnull(val):
        return 'Unknown'
    val_lower = str(val).lower().strip()

    if any(x in val_lower for x in ['fod', 'foood', 'ffood','fodo', 'Foo', 'Foodd','ood', 'oFod']):
        return 'Food'
    elif any(x in val_lower for x in ['rnt', 'ret', 'rentt', 'reent', 'retn', 'ren', 'rnet', 'rennt', 'rnt', 'rrent']):
        return 'Rent'
    elif any(x in val_lower for x in ['tavel', 'trave', 'tarvel' , 'ravel', 'travle', 'rtavel', 'trvael', 'travell', 'trvel', 'traavel', 'travvel', 
                                         'ttravel',  'traevl', 'traveel', 'trravel', 'travl', 'trael', 'traevel']):
        return 'Travel'
    elif any(x in val_lower for x in ['ent', 'netertainment', 'entertanment', 'entertainment', 'etertainment', 'entertainmetn', 'entertaniment', 
                                      'entretainment', 'entertainment', 'entertainemnt', 'enntertainment', 'entertainmnt', 'enteratinment', 
                                      'entertainnment', 'entetainment', 'enterrtainment', 'enttertainment', 'entertianment', 'entertinment', 
                                      'entertaimnent', 'entertainmentt', 'entertainent', 'entertainmnet', 'entertaainment', 'etnertainment', 
                                      'entertaiinment', 'enertainment', 'entetrainment', 'entrtainment', 'enterttainment', 'entertainmet', 
                                      'entertainmennt']):
        return 'Entertainment'
    elif any(x in val_lower for x in ['utilitiees', 'utiilities', 'utiliies', 'utiltiies', 'utilitie', 'utilitise', 'utliities', 'utiilties',
                                       'utilitiess', 'tilities', 'utiliteis', 'utillities', 'utiliities', 'utilties', 'utilitis', 'utilites', 
                                       'utlities', 'uitlities', 'tuilities', 'utilitiies', 'utiities', 'uutilities', 'uilities', 'uttilities']):
        return 'Utilities'
    elif any(x in val_lower for x in [ 'educatino', 'educcation', 'eudcation', 'edcation', 'eduation', 'eeducation',  'educatio', 'educationn', 
                                      'eduucation', 'educatiion', 'educatioon', 'edducation']):
        return 'Education'
    elif any(x in val_lower for x in [ 'heealth', 'heaalth', 'helath', 'healt', 'heath', 'ealth', 'healh']):
        return 'Health'
    elif any(x in val_lower for x in ['asvings', 'savinngs', 'saviings', 'saings', 'svings', 'savings', 'saivngs', 'savigs', 'savins', 'savnigs',
                                      'saavings', 'savvings', 'avings'  , 'ssavings', 'saving'  , 'savngs', 'savinggs', 'savigns', 'savinsg' ]):
        return 'Savings'
    elif any(x in val_lower for x in ['salaryy', 'saary', 'salry', 'aslary', 'alary', 'salaary', 'salray', 'sallary', 'slaary', 'ssalary', 'saalry',
                                       'slary', 'salarry', 'saalary', 'salayr', 'salay',   'salray'  ]):
        return 'Salary'
    elif any(x in val_lower for x in ['bonnus', 'bonu', 'obnus', 'bbonus', 'bouns', 'bnous', 'onus', 'boonus', 'bonuus', 'bons'  ,'bonsu',
                                       'bous', 'bnus' ]):
        return 'Bonus'
    elif any(x in val_lower for x in ['oter income', 'toher income', 'othr income', 'othre income', 'other inncome', 'otehr income', 'othe income', 
                                      'other income', 'other icnome' , 'other  income', 'othe rincome', 'otherincome', 'other incoem',  
                                      'other incom', 'other nicome', 'oother income', 'other iincome', 'other inome', 'ohter income',  
                                      'other icome'  ,    'other incmoe' ,    'oher income'  ,    'other incoe' ]):
        return 'Other Income'
    elif any(x in val_lower for x in ['thers', 'othhers', 'othres', 'otherrs', 'ohters', 'otehrs', 'oothers', 'oters', 'otthers', 
                                         'other', 'othrs', 'othes', 'othesr', 'tohers', 'ohers', 'otherss', 'misc']):
        return 'Others'
    elif val_lower in ['investment', 'freelance']:
        return val.title()
    else:
        return 'Others'
    
df['category'] = df['category'].apply(standardise_category)

print('\nUnique category count after cleaning:', df['category'].nunique())
print('\nCategory values after cleaning:')
print(df['category'].unique())


Unique category count after cleaning: 12

Category values after cleaning:
['Rent' 'Food' 'Entertainment' 'Savings' 'Education' 'Others' 'Salary'
 'Other Income' 'Travel' 'Bonus' 'Health' 'Utilities']


In [83]:
print('\nUnique category count after cleaning:', df['category'].nunique())
print('\nCategory values after cleaning:')
print(df['category'].value_counts().to_string())


Unique category count after cleaning: 12

Category values after cleaning:
category
Food             3558
Rent             2964
Travel           1431
Utilities        1241
Entertainment    1044
Others            751
Bonus             723
Salary            723
Other Income      703
Savings           573
Education         324
Health            224


In [84]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bengaluru,NaN,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,NaN,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
...,...,...,...,...,...,...,...,...,...,...,...,...
15830,T04308,U052,2020-10-15,Expense,Food,140,Cash,Bengaluru,NaN,2020,10,October
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office,2019,6,June
15832,T10743,U001,2022-01-11,Expense,Food,432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta,2022,1,January
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit,2022,12,December


## Step 9: Clean the Payment Mode Column
**Issues:** 60+ variations of just 4 valid values with typos, missing letters, doubled letters

In [85]:
# Strip whitespace first

df['payment_mode'] = df['payment_mode'].str.strip()

In [86]:
df['payment_mode'].unique()

array(['Cash', 'Card', 'UPI', 'Bank Transfer', nan, 'CCash', 'Caard',
       'aCrd', 'Cadr', 'UIP', 'PI', 'Csah', 'Crd', 'Bank Trasnfer',
       'aBnk Transfer', 'Crad', 'Cashh', 'Bnk Transfer', 'BankT ransfer',
       'Cardd', 'ash', 'aCsh', 'Bak Transfer', 'ard', 'Bank Transffer',
       'UUPI', 'UP', 'Csh', 'Car', 'UI', 'PUI', 'BankTransfer',
       'Bank Trransfer', 'Cas', 'Bank Trnsfer', 'Ban Transfer', 'Cah',
       'UPII', 'Bankk Transfer', 'Carrd', 'UPPI', 'Caash', 'CCard',
       'Cassh', 'Bank Transfr', 'Bank Traansfer', 'Baank Transfer',
       'Bank Trannsfer', 'Bank Trasfer', 'Bank Trnasfer',
       'Bank Transsfer', 'Bank Transferr', 'Bank TTransfer',
       'Bank Transfe', 'Cad', 'Bank Transer', 'Bank Tranfser', 'Cahs',
       'Bank rTansfer', 'Bank Tarnsfer', 'Bank Transefr', 'ank Transfer',
       'Bank Transfeer'], dtype=object)

In [87]:
# Use fuzzy keyword matching to catch all variations
def standardise_payment(val):
    if pd.isnull(val):
        return 'Unknown'
    val_lower = str(val).lower().strip()

    if any(x in val_lower for x in ['abnk transfer', 'bnk transfer', 'bankt ransfer', 'bak transfer', 'bank transffer', 'banktransfer',
                                    'bank trransfer', 'bank trnsfer', 'ban transfer',  'bankk transfer',  'bank transfr', 'bank traansfer',
                                    'baank transfer', 'bank trannsfer', 'bank trasfer', 'bank trnasfer', 'bank transsfer', 'bank transferr', 
                                    'bank ttransfer', 'bank transfe',  'bank transer', 'bank tranfser',  'bank rtansfer', 'bank tarnsfer',
                                    'bank transefr', 'ank transfer', 'bank transfeer']):
        return 'Bank Transfer'
    elif any(x in val_lower for x in [ 'ccash', 'csah', 'cashh', 'ash', 'acsh', 'csh', 'cas', 'cah', 'caash', 'cassh', 'cahs']):
        return 'Cash'
    elif any(x in val_lower for x in ['uip', 'pi', 'uupi', 'up',  'ui', 'pui',  'upii',  'uppi']):
        return 'UPI'
    elif any(x in val_lower for x in ['caard', 'acrd', 'cadr', 'crd',  'crad', 'cardd',  'ard', 'car', 'carrd', 'ccard', 'cad']):
        return 'Card'
    else:
        return 'Unknown'
    
df['payment_mode'] = df['payment_mode'].apply(standardise_payment)

df['payment_mode'].unique()

array(['Cash', 'Card', 'UPI', 'Bank Transfer', 'Unknown'], dtype=object)

In [88]:

print('\nUnique payment mode count after cleaning:', df['payment_mode'].nunique())
print('\nPayment mode values after cleaning:')
print(df['payment_mode'].value_counts().to_string())


Unique payment mode count after cleaning: 5

Payment mode values after cleaning:
payment_mode
Bank Transfer    3505
Cash             3472
UPI              3416
Card             3404
Unknown           462


In [89]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bengaluru,NaN,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,NaN,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
...,...,...,...,...,...,...,...,...,...,...,...,...
15830,T04308,U052,2020-10-15,Expense,Food,140,Cash,Bengaluru,NaN,2020,10,October
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office,2019,6,June
15832,T10743,U001,2022-01-11,Expense,Food,432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta,2022,1,January
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit,2022,12,December


## Step 10: Clean the Location Column
**Issues:** Mixed casing, all caps variations like `KOLKATA`, `kolkata`, `Kolkata`, `Bengaluru` vs `Bangalore`

In [90]:
df['location'].unique()

array(['Pune', 'Delhi', 'Bengaluru', nan, 'Hyderabad', 'Kolkata',
       'Ahmedabad', 'Chennai', 'jaipur', 'Lucknow', 'Mumbai', 'mumbai',
       'Jaipur', 'bengaluru', 'delhi', 'pune', 'JAIPUR', 'hyderabad',
       'lucknow', 'DELHI', 'AHMEDABAD', 'ahmedabad', 'HYDERABAD',
       'CHENNAI', 'chennai', 'kolkata', 'PUNE', 'BENGALURU', 'MUMBAI',
       'KOLKATA', 'LUCKNOW'], dtype=object)

In [91]:
# Standardise to title case and strip whitespace
df['location'] = df['location'].str.strip().str.title()

In [92]:
df['location'].unique()

array(['Pune', 'Delhi', 'Bengaluru', nan, 'Hyderabad', 'Kolkata',
       'Ahmedabad', 'Chennai', 'Jaipur', 'Lucknow', 'Mumbai'],
      dtype=object)

In [93]:
# Map all variations to standard city names
location_mapping = {
    'Bengaluru'  : 'Bangalore',
    'Bangalore'  : 'Bangalore',
    'Mumbai'     : 'Mumbai',
    'Delhi'      : 'Delhi',
    'Hyderabad'  : 'Hyderabad',
    'Chennai'    : 'Chennai',
    'Kolkata'    : 'Kolkata',
    'Pune'       : 'Pune',
    'Ahmedabad'  : 'Ahmedabad',
    'Jaipur'     : 'Jaipur',
    'Lucknow'    : 'Lucknow',
}

df['location'] = df['location'].map(location_mapping)

df['location'] = df['location'].fillna('Unknown')

In [94]:
print('\nLocation values after cleaning:')
print(df['location'].value_counts().to_string())


Location values after cleaning:
location
Hyderabad    1427
Pune         1394
Bangalore    1393
Mumbai       1371
Jaipur       1367
Chennai      1356
Kolkata      1355
Delhi        1344
Ahmedabad    1320
Lucknow      1284
Unknown       648


In [95]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bangalore,NaN,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,Unknown,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
...,...,...,...,...,...,...,...,...,...,...,...,...
15830,T04308,U052,2020-10-15,Expense,Food,140,Cash,Bangalore,NaN,2020,10,October
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office,2019,6,June
15832,T10743,U001,2022-01-11,Expense,Food,432,Cash,Ahmedabad,Aw5aDyInfbtc9hQta,2022,1,January
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit,2022,12,December


## Step 11: Clean the Notes Column
**Issues:** Random alphanumeric junk strings like `hZtATyn1UX55solCMr1`, missing values

In [96]:
print('Notes sample before cleaning:')
print(df['notes'].value_counts(dropna=False).head(15))

Notes sample before cleaning:
notes
NaN                      1383
Gift                      886
Doctor visit              859
Dinner at resto           853
Netflix subscription      839
Monthly rent              835
Uber to office            829
Paid electricity bill     824
Grocery shopping          811
Course fee                798
Salary                    797
Doctor visit via app      114
Course fee                107
Grocery shopping          105
Salary via app            104
Name: count, dtype: int64


In [97]:
# Strip whitespace

df['notes'] = df['notes'].str.strip()

In [98]:
# Replace junk alphanumeric strings — random mix of letters and numbers
junk_pattern = r'^(?=.*[0-9])(?=.*[a-zA-Z])[a-zA-Z0-9]{8,}$'

df['notes'] = df['notes'].replace(junk_pattern, 'N/A', regex=True)

In [99]:
# Fill missing notes with N/A

df['notes'] = df['notes'].fillna('N/A')

In [100]:
print('\nNotes sample after cleaning:')
print(df['notes'].value_counts().head(15))


Notes sample after cleaning:
notes
N/A                           2181
Gift                           981
Dinner at resto                955
Doctor visit                   946
Netflix subscription           936
Monthly rent                   932
Paid electricity bill          920
Uber to office                 916
Grocery shopping               916
Course fee                     905
Salary                         895
Doctor visit via app           114
Salary via app                 104
Uber to office cash            101
Paid electricity bill late     101
Name: count, dtype: int64


In [101]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bangalore,N/A,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,Unknown,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
...,...,...,...,...,...,...,...,...,...,...,...,...
15830,T04308,U052,2020-10-15,Expense,Food,140,Cash,Bangalore,N/A,2020,10,October
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office,2019,6,June
15832,T10743,U001,2022-01-11,Expense,Food,432,Cash,Ahmedabad,N/A,2022,1,January
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit,2022,12,December


## Step 12 : Update Data Types

In [102]:
df.dtypes

transaction_id      object
user_id             object
date                object
transaction_type    object
category            object
amount               int64
payment_mode        object
location            object
notes               object
year                 int32
month                int32
month_name          object
dtype: object

In [103]:
# Convert date to datetime

df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Convert low cardinality text columns to category dtype
df['transaction_type'] = df['transaction_type'].astype('category')
df['category']         = df['category'].astype('category')
df['payment_mode']     = df['payment_mode'].astype('category')
df['location']         = df['location'].astype('category')

print('--- Updated Data Types ---')
print(df.dtypes)

--- Updated Data Types ---
transaction_id              object
user_id                     object
date                datetime64[ns]
transaction_type          category
category                  category
amount                       int64
payment_mode              category
location                  category
notes                       object
year                         int32
month                        int32
month_name                  object
dtype: object


In [104]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bangalore,N/A,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,Unknown,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
...,...,...,...,...,...,...,...,...,...,...,...,...
15830,T04308,U052,2020-10-15,Expense,Food,140,Cash,Bangalore,N/A,2020,10,October
15831,T05700,U101,2019-06-15,Expense,Food,470,UPI,Pune,Uber to office,2019,6,June
15832,T10743,U001,2022-01-11,Expense,Food,432,Cash,Ahmedabad,N/A,2022,1,January
15833,T00538,U125,2022-12-17,Expense,Food,84,UPI,Kolkata,Doctor visit,2022,12,December


## Step 13: Final Validation Check

In [105]:
print(' Final Shape ')
print(df.shape)

print('\n Final Missing Values ')
print(df.isnull().sum())

print('\n Final Category Values ')
print(df['category'].value_counts())

print('\n Final Payment Mode Values ')
print(df['payment_mode'].value_counts())

print('\n Final Location Values ')
print(df['location'].value_counts())

print('\n Final Amount Stats ')
print(df['amount'].describe())

print('\n Final Sample ')
df.head(10)

 Final Shape 
(14259, 12)

 Final Missing Values 
transaction_id      0
user_id             0
date                0
transaction_type    0
category            0
amount              0
payment_mode        0
location            0
notes               0
year                0
month               0
month_name          0
dtype: int64

 Final Category Values 
category
Food             3558
Rent             2964
Travel           1431
Utilities        1241
Entertainment    1044
Others            751
Salary            723
Bonus             723
Other Income      703
Savings           573
Education         324
Health            224
Name: count, dtype: int64

 Final Payment Mode Values 
payment_mode
Bank Transfer    3505
Cash             3472
UPI              3416
Card             3404
Unknown           462
Name: count, dtype: int64

 Final Location Values 
location
Hyderabad    1427
Pune         1394
Bangalore    1393
Mumbai       1371
Jaipur       1367
Chennai      1356
Kolkata      1355
Delhi      

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T03512,U039,2021-12-22,Expense,Rent,998,Cash,Pune,Paid electricity bill,2021,12,December
1,T03261,U179,2022-03-24,Expense,Food,143,Card,Delhi,Grocery shopping,2022,3,March
2,T04316,U143,2022-10-18,Expense,Rent,149,Cash,Bangalore,N/A,2022,10,October
3,T05649,U079,2021-12-12,Expense,Rent,49,UPI,Unknown,Paid electricity bill,2021,12,December
5,T04246,U188,2022-07-12,Expense,Entertainment,208,UPI,Hyderabad,Monthly rent,2022,7,July
6,T01814,U150,2019-09-12,Expense,Savings,681,Cash,Kolkata,Salary,2019,9,September
7,T06293,U014,2022-01-06,Expense,Rent,2524,Bank Transfer,Unknown,Salary,2022,1,January
8,T07930,U138,2022-11-04,Expense,Education,93,Bank Transfer,Pune,N/A,2022,11,November
9,T04656,U112,2021-02-03,Expense,Food,323,UPI,Ahmedabad,Monthly rent,2021,2,February
10,T11712,U117,2022-11-02,Expense,Others,662,UPI,Chennai,Salary cash,2022,11,November


## Step 14: Export Clean Dataset

In [ ]:
df.to_csv('cleaned_synthetic.csv', index=False)

## Step 15: Loading Cleaned Dataset into Postgresql

In [106]:
from sqlalchemy import create_engine
from sqlalchemy import text

# -----------------------------------------------
# PostgreSQL Connection Settings
# -----------------------------------------------
DB_USER     = 'postgres'
DB_PASSWORD = 'Chibueze04'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'budgetwise_db'

# Create connection engine
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Drop the view first so the table can be replaced
with engine.connect() as conn:
    conn.execute(text('DROP VIEW IF EXISTS master_transactions;'))
    conn.commit()
    print('View dropped successfully')
    
# Load clean dataframe into PostgreSQL
df.to_sql(
    name      = 'budgetwise_synthetic',
    con       = engine,
    if_exists = 'replace',
    index     = False
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text('SELECT COUNT(*) FROM budgetwise_synthetic'))
    print('Rows in budgetwise_synthetic:', result.scalar())

print('budgetwise_synthetic loaded into PostgreSQL successfully')

View dropped successfully
Rows in budgetwise_synthetic: 14259
budgetwise_synthetic loaded into PostgreSQL successfully
